# Predicting Melting Points with NVIDIA ALCHEMI Toolkit

Direct coexistence (solid–liquid) molecular dynamics on naphthalene using the AIMNet2-2025 foundation model.

> **Note.** The notebook ships with `FAST_DEMO=True`: analysis cells read from cached trajectories under `assets/`, so the full pipeline can be walked through in minutes. Flip `FAST_DEMO=False` (cell 11) to reproduce the cache live on a GPU.

## Motivation

### Why melting points matter

The melting point $T_\mathrm{m}$ is one of the most basic — and most stubborn — predictions in molecular materials. It governs the processability of pharmaceuticals, the storage stability of energetic materials, the casting window of polymers, and the operating range of thermal-management fluids. A workflow that takes a crystal structure and returns a reliable $T_\mathrm{m}$ could potentially compress weeks of wet-lab thermal characterisation.

### The direct-coexistence approach

Single-phase NPT melting massively superheats: a perfect crystal at ambient pressure can persist hundreds of kelvins above its true $T_\mathrm{m}$ before nucleating a liquid. The **direct-coexistence (SLC) method** sidesteps the nucleation barrier by stitching a slab of solid to a slab of liquid in the same simulation cell. At $T_\mathrm{m}$ the interface is stationary; below it, the solid grows; above it, the solid melts. The procedure followed here is the SLC + rotational-order-parameter screening recipe of Schmidt, Van der Spoel & Walz [1].

### ALCHEMI Toolkit framing

The NVIDIA ALCHEMI Toolkit ships GPU-accelerated MD integrators (NVT-Langevin, anisotropic NPT, FIRE optimizers), foundation-model wrappers (e.g. MACE and AIMNet2 variants), and the `SnapshotHook` / `LoggingHook` infrastructure used throughout this notebook. Two design decisions distinguish it from a typical MLIP-on-ASE workflow: integrators consume a multi-graph **`Batch`** rather than one system at a time (so the 5-temperature SLC sweep in §9 runs as a single GPU launch), and per-step lifecycle behaviour is composed from **hooks** (safety, logging, snapshots, convergence) rather than monkey-patched into the integrator. We will combine those primitives to walk through the full SLC pipeline on naphthalene — from a CIF file to a temperature sweep that brackets the experimental $T_\mathrm{m,exp} = 353$ K.

## What You Will Learn

| # | Section | What you will do |
|---|---------|------------------|
| 1 | Environment Setup | Configure runtime knobs, the **FAST_DEMO** toggle, and import the toolkit primitives. |
| 2 | Crystal Structure | Read the experimental naphthalene CIF, build a supercell, validate the cell against the foundation-model cutoff. |
| 3 | Foundation Model | Load `AIMNet2Wrapper.from_checkpoint("aimnet2_2025")` and configure its active outputs for energy, forces, and stress. |
| 4 | Warmup MD | Three-stage equilibration: FIRE2 minimisation → NVT thermalisation → anisotropic NPT @ $T_\mathrm{warmup}$. |
| 5 | Warmup Diagnostics | Density plateau, COM-MSD plateau, rotational $S_0$. Read the **Yoneya–Harada phase classifier**. |
| 6 | Liquid Half | Reseed velocities at $T_\mathrm{melt}$ and run NVT to generate the molten counterpart. |
| 7 | SLC Construction | Stack solid + liquid along the monoclinic unique axis, insert a vacuum gap, unwrap molecules. |
| 8 | SLC Pre-equilibration | FIRE2 + short NVT at each target temperature (Yoneya–Harada procedure). |
| 9 | SLC Production | Anisotropic NPT across a temperature sweep as one multi-graph batch. |
| 10 | $T_\mathrm{m}$ Extraction | Per-temperature density, MSD, and $S_0$. Identify the bracket that contains $T_\mathrm{m}$. |

### Prerequisites

| Requirement | Description |
|-------------|-------------|
| Python ≥ 3.11 | Container ships a virtual environment using `uv`, with `nvalchemi-toolkit` installed. |
| GPU | A single CUDA GPU is enough for `FAST_DEMO=True` (analysis only). For `FAST_DEMO=False`, expect ~6 hours of A100/H100 wall time end-to-end. |
| Cached data | `assets/naphthalene_long_2025/traj/*.extxyz` ship with the container and feed the FAST_DEMO path. |
| Background | Basic familiarity with the Atomic Simulation Environment (`ase`) package, as well as concepts in MD such as thermostats, barostats, and NVT/NPT ensembles would be helpful, but is not required. No prior experience with the ALCHEMI Toolkit package is required, you are here to learn. |

### AIMNet2 / AIMNet2-2025

[AIMNet2](https://github.com/zubatyuk/aimnet2) is a charge-equivariant neural-network potential trained on $\omega$B97M-D3 reference data for organic molecules [3]; **AIMNet2-2025** is a reparameterized variant (B97-3c+D3) that offers improved intermolecular accuracy (vital for crystalline structures). Both checkpoints can be loaded via toolkit's `AIMNet2Wrapper`.

Two practical notes shape the rest of this notebook:

1. **Cutoff**: the model uses a 5 Å radial cutoff. Long-range electrostatics are not modelled by the bare network. For weak-dipole compounds like naphthalene this is workable; for polar molecular crystals an Ewald wrapper is recommended (we discuss this in *Extensions*).
2. **Anisotropic stress**: AIMNet2-2025 supports stress via autograd when ``set_config("active_outputs", {"energy", "forces", "stress"})`` is called. This is what enables the anisotropic NPT integrator to read a real per-axis pressure — essential for the SLC geometry (§7).

### Direct-Coexistence (SLC)

Place a slab of equilibrated **solid** next to a slab of equilibrated **liquid** in the same periodic box, with the interface normal aligned to one of the lattice axes. Run NPT at a target $T$:

$$
\text{interface motion} =
\begin{cases}
\text{liquid freezes onto the solid (interface advances)} & T < T_\mathrm{m} \\
\text{solid melts (interface retreats)} & T > T_\mathrm{m} \\
\text{stationary — both phases coexist} & T = T_\mathrm{m}.
\end{cases}
$$

Schmidt, Van der Spoel & Walz [1] sweep the production NPT across **multiple target temperatures** and read $T_\mathrm{m}$ from the temperature at which the rotational order parameter $S_0$ and the translational diffusion coefficient $D$ jointly cross from solid- to liquid-like values (see §10). To stabilise the stitched geometry before the production NPT we follow their pre-equilibration recipe — a short FIRE minimisation followed by a brief NVT thermalisation per target temperature (§8).

### Order parameters: $S_0$, COM-MSD, and the phase classifier

We diagnose phase via two independent quantities:

**Translational diffusion** $D$ (Einstein relation, 3D):

$$\langle |\vec r_\mathrm{COM}(t) - \vec r_\mathrm{COM}(0)|^2 \rangle = 6 D t.$$

We fit the slope of the molecular-COM mean-squared displacement vs time over the trailing fraction of the trajectory [4]. Under anisotropic NPT a few subtleties (affine cell-deformation removal, system-COM subtraction) are needed to avoid spurious diffusion; §5 walks through them.

**Rotational order** $S_0$ [1]: the tail-average of the $P_2$ orientational autocorrelation function of each molecule's three principal-inertia axes:

$$S_0 = \langle \tfrac{1}{2}(3\cos^2\theta_i(t) - 1)\rangle_{i,\,t \to \infty}.$$

$S_0 = 1$ means no rotation (perfect orientational order); $S_0 = 0$ means free rotation (isotropic liquid); intermediate values indicate hindered rotation. Combining $S_0$ with $D$ yields the phase classifier:

| Phase | $D$ | $S_0$ |
|-------|-----|-------|
| Crystal | $\approx 0$ | $\to 1$ |
| Plastic crystal | $\approx 0$ | $0 < S_0 < 1$ |
| Liquid | $\gg 0$ | $\to 0$ |

We use this as the through-line in every diagnostic section of the notebook.

### Pipeline overview

```
  CIF (Brock & Dunitz 1982 [2])
       │
       ▼
  ASE supercell (5,5,4 = 200 mol = 3600 atoms)
       │
       ▼
  AIMNet2-2025 + safety hooks
       │
       ▼
  ┌──────────────── Warmup (single-system) ──────────────┐
  │ FIRE2 minimise  →  NVT @ T_warmup  →  NPT @ T_warmup │
  └──────────────────────────────────────────────────────┘
       │
       ▼
  ┌──────── Melt generator ────────┐
  │  reseed v ~ Maxwell–Boltzmann  │
  │  at T_melt → NVT @ T_melt      │
  └────────────────────────────────┘
       │
       ▼
  ┌────────── SLC construction ─────────────┐
  │  stack solid + melt along lattice vector│
  │  insert vacuum gap, account for PBC     │
  └─────────────────────────────────────────┘
       │
       ▼
  ┌────────────── SLC sweep (multi-graph: T ∈ TEMPS) ────────────┐
  │  FIRE2  →  NVT @ T (10 ps)  →  anisotropic NPT @ T (200 ps)  │
  └──────────────────────────────────────────────────────────────┘
       │
       ▼
  Per-T density, MSD, S_0  →  T_m bracket
```

> **Scope note.** This notebook is a single-composition demonstration of the SLC method, not a quantitative $T_\mathrm{m}$ benchmark. The default config uses the AIMNet2-2025 model, a 200-molecule supercell, and a 5-temperature sweep — all chosen so the full pipeline can be recomputed in a brief amount of time. For reference, Schmidt et al. [1] used the classical GAFF force field, supercells of $\geq 1000$ molecules per compound, and $\geq 15$ ns of NPT equilibration per system across a benchmark of 30 organic crystals. See *Extensions* for the path forward.

## 1. Environment Setup

In [6]:
# ─── Package versions ──────────────────────────────────────────────────────
from importlib.metadata import version, PackageNotFoundError

for pkg in ["nvalchemi-toolkit", "torch", "ase", "numpy", "matplotlib", "tqdm"]:
    try:
        print(f"{pkg:22s} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:22s} (not installed)")

nvalchemi-toolkit      0.1.0
torch                  2.11.0+cu130
ase                    3.28.0
numpy                  2.4.4
matplotlib             3.10.9
tqdm                   4.67.3


In [ ]:
# ─── Control Panel ─────────────────────────────────────────────────────────
# Knobs for the entire pipeline. Detailed motivation appears alongside
# first use; defaults below match the cached `naphthalene_long_2025` run.

# === Run identity ===========================================================
# Flip FAST_DEMO=False to run the full MD pipeline live on a GPU container.
# FAST_DEMO=True (default) skips integrator runs and feeds analysis cells
# from cached extended-XYZ trajectories under assets/<RUN_NAME>/traj/.
FAST_DEMO = True
RUN_NAME = "naphthalene_long_2025"

# === Physical parameters ====================================================
# Temperatures are chosen relative to TM_EXP: T_WARMUP ≪ TM_EXP (crystal
# survives), T_MELT ≫ TM_EXP (unambiguous liquid), TEMPS spans both.
TM_EXP = 353.0  # K  — naphthalene experimental melting point
T_WARMUP = 100.0  # K  — warmup target (NVT + NPT)
T_MELT = 500.0  # K  — melt-generator target
TEMPS = (250.0, 300.0, 350.0, 400.0, 450.0)  # SLC sweep (K), brackets TM_EXP
SUPERCELL = (5, 5, 4)  # 200 molecules = 3600 atoms
MELT_SRC = "npt"  # which warmup endpoint feeds melt + SLC ("npt" or "nvt")

# === Foundation model =======================================================
MODEL_NAME = "aimnet2_2025"  # AIMNet2 B97-3c+D3 reparameterization

# === MD numerics ============================================================
# The thermostat / barostat coupling times below are introduced alongside
# the integrators in §4.
DT = 0.5  # fs — MD timestep
FRICTION = 0.01  # fs⁻¹ — Langevin friction
THERMOSTAT_TIME = 100.0  # fs — Nosé–Hoover chain coupling time
BAROSTAT_TIME_FS = 5000.0  # fs — MTK τ_P; long enough to decouple from interface motion
FMAX = 0.15  # eV/Å — FIRE2 force convergence threshold
FIRE_MAX_STEPS = 5000

# === Production durations (only consumed when FAST_DEMO=False) ==============
# These match the parameters used to generate the cached
# `naphthalene_long_2025` trajectories shipped in assets/.
THERMALIZE_PS = 15.0  # warmup NVT
EQUILIBRATE_PS = 50.0  # warmup NPT
MELT_PS = 15.0
EQUIL_SLC_PS = 10.0  # per-T SLC pre-equilibration NVT
SLC_PS = 200.0  # production NPT

# === Logging cadence (steps between snapshots / CSV rows) ===================
SNAPSHOT_EVERY = 100
LOG_EVERY = 100

### Imports

Throughout the notebook we will import individual helpers from the `helpers/` package **just-in-time** in the cell that uses them, with a one-line description of what each helper does. The cell below pulls in only the **library-level** dependencies (ASE, PyTorch, the ALCHEMI Toolkit core integrators, and `numpy` / `matplotlib`) plus a handful of toolkit-wide settings. Helpers that expose interesting toolkit patterns (the safety-hook chain, the LoggingHook custom-scalar protocol, etc.) we will **define inline** alongside their use sites for you to gain familiarity with ALCHEMI Toolkit.

In [ ]:
# ─── Library imports ──────────────────────────────────────────────────────
import csv
from loguru import logger
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from ase.io import read as ase_read

# Disable donated-buffer optimization so torch.compile coexists with the
# pipeline's retain_graph=True stress autograd. Only relevant under live MD
# (compile_model=True in §3). torch._functorch.config is lazy-loaded in newer
# torch builds, hence the explicit import.
try:
    import torch._functorch.config  # noqa: F401
    torch._functorch.config.donated_buffer = False
except (ImportError, AttributeError):
    pass
torch.set_float32_matmul_precision("high")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Tag construction (these match the artefact stems produced by the
# warmup / melt / slc drivers, which lets us cross-reference cached
# trajectories under assets/<RUN_NAME>/traj/).
DT_TAG = f"dt{str(DT).replace('.', 'p')}fs"  # e.g. 'dt0p5fs'
T_WARMUP_TAG = f"{int(T_WARMUP)}k"  # e.g. '100k'
T_MELT_TAG = f"{int(T_MELT)}k"  # e.g. '500k'
ASSETS_DIR = Path("assets") / RUN_NAME
TRAJ_DIR = ASSETS_DIR / "traj"
LOGS_CACHE_DIR = ASSETS_DIR / "logs"
LOG_DIR = Path("logs") / RUN_NAME
LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run {RUN_NAME!r} on {DEVICE} | T_warmup={T_WARMUP} K | FAST_DEMO={FAST_DEMO}")
print(f"Tags: {T_WARMUP_TAG}, {DT_TAG}")
print(f"Cached trajectories: {TRAJ_DIR}")

### Notebook-local utilities

Two small functions for the cached-trajectory pathway used by the FAST_DEMO branch:

- **`ase_frames_to_batches`** maps `ase.io.read(path, index=':')` output to a list of single-graph `Batch` objects ready for the analysis helpers (`compute_S0_from_frames`, `compute_msd`, `compute_com_msd`). It is a thin wrapper around `AtomicData.from_atoms` plus zero-allocated dynamics fields, so each frame is a self-contained dynamics-ready record. This lets the FAST_DEMO branch feed the diagnostics indistinguishably from a live MD run.

- **`cached_extxyz`** resolves a stage-stem to its cached extxyz path under `assets/<RUN_NAME>/traj/`, returning `None` if the file is missing. Diagnostic cells call it to short-circuit the live MD.

- **`cached_log_csv`** is its sibling for `LoggingHook` CSVs under `assets/<RUN_NAME>/logs/`. FAST_DEMO mode opens these and replays them through `stdout_writer` (defined in §4) so each pipeline cell prints the same per-step trace a live MD run would produce — same look, no live compute.

(`AtomicData` and `Batch` themselves are introduced in §2, where we build the first one from a real CIF.)

In [ ]:
# ─── Notebook-local utilities ──────────────────────────────────────────────
# AtomicData / Batch are the toolkit's two core data containers; we
# introduce them in §2 below. AtomicData.from_atoms (mirrored from the
# toolkit's basic/03_ase_integration.py example) is the canonical
# ASE → toolkit converter — it reads positions / atomic_numbers / cell /
# pbc straight off any ASE Atoms object.
from nvalchemi.data import AtomicData, Batch


def ase_frames_to_batches(frames, device="cpu"):
    """Convert iterable of ase.Atoms (e.g. ase.io.read(path, index=':')) into
    a list of single-graph Batch objects, ready for the helpers/analysis
    diagnostics. Velocities are zero — cached frames carry no momenta and
    none of the post-hoc analysers consume them."""
    batches = []
    for atoms in frames:
        data = AtomicData.from_atoms(atoms)
        n = data.num_nodes
        data.forces = torch.zeros(n, 3)
        data.energy = torch.zeros(1, 1)
        data.stress = torch.zeros(1, 3, 3)
        data.add_node_property("velocities", torch.zeros(n, 3))
        data.charge = torch.zeros(1, 1)
        batches.append(Batch.from_data_list([data], device=device))
    return batches


def cached_extxyz(stem):
    """Resolve assets/<RUN_NAME>/traj/<stem>.extxyz; return None if absent."""
    path = TRAJ_DIR / f"{stem}.extxyz"
    return path if path.exists() else None


def cached_log_csv(stem):
    """Resolve assets/<RUN_NAME>/logs/<stem>.csv; return None if absent."""
    path = LOGS_CACHE_DIR / f"{stem}.csv"
    return path if path.exists() else None


## 2. Crystal Structure

Naphthalene ($\mathrm{C_{10}H_{8}}$) is the simplest fused-aromatic molecule and the textbook example of a monoclinic molecular crystal. It crystallises in space group $P2_1/a$ (#14) with $Z = 2$ molecules per unit cell, packed in a herringbone motif. We use the room-temperature reference structure of Brock & Dunitz (CCDC 1216816, refcode `NAPHTA10`) [2]; ASE reads the CIF directly.

### `AtomicData` and `Batch` — the toolkit's data containers

The toolkit's two core data structures appear in every cell that touches positions or forces:

- **`AtomicData`** is the toolkit's equivalent of ASE's `Atoms` object — a single-system record holding `positions`, `atomic_numbers`, `cell`, `pbc`, and the per-step dynamics fields (`forces`, `energy`, `stress`, `velocities`). Conversion from ASE is one call: `AtomicData.from_atoms(atoms)` populates everything topology-related and leaves the dynamics fields for the caller to allocate.

- **`Batch`** collates one or more `AtomicData` records into the multi-graph object every integrator and hook in the toolkit consumes. Even a single system is wrapped in a length-1 `Batch` (e.g. our initial supercell below); §9 stacks the same mechanism into a 5-temperature SLC sweep.

We saw the import in §1 (`from nvalchemi.data import AtomicData, Batch`) so that `ase_frames_to_batches` had access to both. The first time we actually build one from real data is the cell below — read the CIF, build a supercell, then construct `AtomicData` + `Batch` from it.

In [ ]:
# ─── CIF read ──────────────────────────────────────────────────────────────
# ASE 3.28's CIF parser flags 'monoclinic' with a UserWarning if it can't
# uniquely map to the P2_1/a vs P2_1/c vs P2_1/n setting. NAPHTA10 supplies
# explicit symmetry operators via _symmetry_equiv_pos_as_xyz, so the
# warning is cosmetic and we let it through.
unit_cell = ase_read("data/naphthalene.cif")
ATOMS_PER_MOL = len(unit_cell) // 2  # Z=2 for P2_1/a

a, b, c = unit_cell.cell.lengths()
alpha, beta, gamma = unit_cell.cell.angles()
print(f"Unit cell: {len(unit_cell)} atoms ({ATOMS_PER_MOL} atoms/molecule)")
print(f"Cell: a={a:.3f}  b={b:.3f}  c={c:.3f}  beta={beta:.1f}°")

In [ ]:
# Build the supercell from the unit cell read above.
supercell = unit_cell * SUPERCELL
N_MOL = len(supercell) // ATOMS_PER_MOL
n_atoms_total = len(supercell)
print(f"Supercell {SUPERCELL}: {n_atoms_total} atoms ({N_MOL} molecules)")

### A density helper

Before we read the CIF we'll define a small helper that comes up in every diagnostic section: per-graph **density** in g/cm³. `nvalchemi` stores atomic masses in atomic mass units and cell vectors in Å, so we need a unit conversion: 1 amu/Å³ = 1.66054 g/cm³.

In [10]:
# ─── Per-graph density ─────────────────────────────────────────────────────
# A common pattern when working with a multi-graph Batch (e.g. the 5
# temperatures in §9): aggregate a per-atom quantity into one number per
# graph using torch's scatter_add_ over batch.batch_idx. compute_density
# does exactly that for total mass, then divides by per-graph cell volume.

AMU_OVER_A3_TO_G_CM3 = 1.66054  # 1 amu/A^3 in g/cm^3


def compute_density(batch):
    """Per-graph density (g/cm^3). Returns list[float] of length num_graphs."""
    vol = torch.linalg.det(batch.cell).abs()  # [num_graphs]
    mass = torch.zeros(batch.num_graphs, device=vol.device)
    mass.scatter_add_(0, batch.batch_idx, batch.atomic_masses)  # sum per graph
    return (mass * AMU_OVER_A3_TO_G_CM3 / vol).tolist()

In [ ]:
# AtomicData.from_atoms reads positions / atomic_numbers / cell / pbc off
# the ASE supercell; the dynamics fields (forces / energy / stress /
# velocities) are pre-allocated as zeros so the integrator can write into
# them in place every step. atomic_masses auto-populates via an
# AtomicData validator. The DEVICE move happens at the Batch step.
data = AtomicData.from_atoms(supercell)
data.forces = torch.zeros(n_atoms_total, 3)
data.energy = torch.zeros(1, 1)
data.stress = torch.zeros(1, 3, 3)
data.add_node_property("velocities", torch.zeros(n_atoms_total, 3))
data.charge = torch.zeros(1, 1)
batch = Batch.from_data_list([data], device=DEVICE)

cell_lengths = batch.cell.squeeze().norm(dim=-1)
print(f"Cell lengths: {[f'{length:.2f}' for length in cell_lengths.tolist()]} A")
print(
    f"Density:      {compute_density(batch)[0]:.3f} g/cm^3 "
    f"(experimental ≈ 1.18 at 298 K)"
)
assert (cell_lengths > 10.0).all(), "Cell too small for AIMNet2 cutoff"


In [ ]:
# ─── Visualise the initial supercell ───────────────────────────────────────
# helpers.visualize_structure renders the (b, c) face of a single-graph
# Batch (or ASE Atoms): per-atom scatter coloured by element with intra-
# molecular bonds drawn as line segments. PBC-aware connectivity discovers
# molecules and unwraps each one before render so none span a periodic
# image — same matplotlib pipeline as render_warmup_s0.py, which animates
# this view over the warmup trajectory coloured by per-molecule S₀
# (here we colour by atom type instead).
from helpers import visualize_structure

visualize_structure(
    batch,
    title=f"Naphthalene {SUPERCELL} supercell ({n_atoms_total} atoms)",
    save_path=str(LOG_DIR / "initial_crystal.png"),
)

## 3. Foundation Model

Load AIMNet2-2025 from its bundled checkpoint. One configuration call matters: `set_config("active_outputs", {"energy", "forces", "stress"})` enables the autograd-stress path. Without this, the NPT integrator's barostat sees only zeros and the cell never relaxes.

For molecular crystals with non-negligible dipoles the bare 5 Å cutoff is insufficient and an Ewald-electrostatics wrapper is needed; we discuss this in *Caveats*.

In [ ]:
# ─── Load AIMNet2-2025 ─────────────────────────────────────────────────────
# AIMNet2Wrapper sits between the raw AIMNet2 checkpoint and the toolkit's
# integrator/hook interfaces — supplies make_neighbor_hooks() (consumed by
# make_safety_hooks below) and from_checkpoint() loading.
from nvalchemi.models.aimnet2 import AIMNet2Wrapper

# In FAST_DEMO mode we skip the model load entirely (no MD will run) and
# substitute None — the analysis cells never touch the model.
if FAST_DEMO:
    aimnet2 = None
    print("FAST_DEMO=True: skipping model load (analysis-only mode).")
else:
    aimnet2 = AIMNet2Wrapper.from_checkpoint(
        MODEL_NAME,
        device=DEVICE,
        compile_model=True,
    )
    aimnet2.set_config("active_outputs", {"energy", "forces", "stress"})
    nc = aimnet2.model_config.neighbor_config
    print(f"AIMNet2-2025 loaded on {DEVICE}, cutoff={nc.cutoff} A, skin={nc.skin}")
    print(f"active_outputs: {sorted(aimnet2.model_config.active_outputs)}")
    eff_cutoff = nc.cutoff + nc.skin
    min_cell_dim = batch.cell.squeeze().norm(dim=-1).min().item()
    if eff_cutoff >= min_cell_dim / 2:
        logger.warning(
            f"Cutoff {eff_cutoff:.2f} A >= half min cell dim "
            f"{min_cell_dim / 2:.2f} A. Neighbor list does redundant work."
        )

## 4. Warmup: FIRE2 → NVT → NPT

Three sub-stages bring the supercell from its experimental geometry to a fully equilibrated MD configuration at the warmup temperature ($T_\mathrm{warmup} = $ `T_WARMUP` K):

**Stage A — FIRE2 minimisation.** A pseudo-dynamics optimiser (force-velocity coupling with adaptive step size) drives forces below ~`FMAX` eV/Å. Note that the FIRE2 `dt` is an **optimizer-internal** step (~0.01), unrelated to the physical MD timestep.

**Stage B — NVT thermalisation.** Maxwell–Boltzmann velocities are seeded at $T_\mathrm{warmup}$ and a Langevin thermostat (friction = `FRICTION` fs⁻¹) draws kinetic energy distributions into thermal equilibrium over `THERMALIZE_PS` ps.

**Stage C — anisotropic NPT.** A Martyna–Tobias–Klein barostat (τ_P = `BAROSTAT_TIME_FS` fs) couples a hydrostatic 1 atm target to each diagonal cell axis independently. The cell relaxes anisotropically under the model's stress tensor; lattice constants converge over `EQUILIBRATE_PS` ps.

### Why anisotropic NPT?

A scalar pressure target with `pressure_coupling="isotropic"` drives all three diagonal cell components against a single hydrostatic value, which is fine for a perfect cubic crystal but imposes spurious lateral strain on a low-symmetry cell. Naphthalene is monoclinic ($\beta = 123.6°$), so the (a,b,c) axes equilibrate to different values under 1 atm. Anisotropic coupling lets each axis track its own component of the pressure tensor:

$$P_\mathrm{target} = \begin{bmatrix} P_0 & P_0 & P_0 \end{bmatrix}, \quad P_0 = 1\ \mathrm{atm}.$$

(In `nvalchemi` this is passed as a `[1, 3]` tensor that broadcasts to every graph in the batch.) Anisotropic NPT is also **mandatory** for the SLC stage in §9, where the interface-normal axis must move independently of the in-plane axes pinned by the solid lattice.

### Hooks — the toolkit's lifecycle callback system

Most of what follows uses **hooks**: callbacks that fire at specific points in the integrator's per-step lifecycle (`BEFORE_STEP`, `BEFORE_COMPUTE`, `AFTER_STEP`, `AFTER_POST_UPDATE`, `ON_CONVERGE`, …). The safety chain, per-step CSV/stdout logging, snapshot writing, and even FIRE2's fmax termination criterion are all hooks. You compose them by passing a list to `hooks=` at integrator construction:

```python
stage = NVTLangevin(model=..., hooks=[<hook>, <hook>, ...])
```

**Order matters** when one hook produces input the next consumes — for example, the neighbour-list rebuild must precede the max-force clamp so the clamp sees fresh edges.

The next two cells define the two recurring hook patterns we'll use throughout the warmup, melt, and SLC pipelines.

### The safety-hook chain

`make_safety_hooks` assembles a defensive hook chain that every integrator below runs with: a **model-supplied neighbour-list rebuild** (different models need different neighbour-list logic, so we delegate to `model.make_neighbor_hooks()`), **periodic-boundary wrapping** (`WrapPeriodicHook`), a **max-force clamp** (`MaxForceClampHook`, guards against rare forward-pass spikes), and a **NaN detector** (`NaNDetectorHook`, fails fast rather than continuing with garbage). The `track_stress` flag toggles whether NaN detection also watches `stress` — turn it off for FIRE2 (no autograd through the stress tape) and on for NVT/NPT.

In [ ]:
# ─── Safety-hook chain ─────────────────────────────────────────────────────
from nvalchemi.dynamics.base import DynamicsStage
from nvalchemi.dynamics.hooks import MaxForceClampHook, NaNDetectorHook
from nvalchemi.hooks import WrapPeriodicHook

MAX_FORCE_CLAMP = 50.0  # eV/A — clamp limit for the safety hook chain


def make_safety_hooks(model, track_stress=True, max_force=MAX_FORCE_CLAMP):
    """Assemble the defensive hook chain.

    Order matters: neighbours rebuild BEFORE the others see new positions;
    periodic-wrapping happens after every position update; force-clamping
    protects against rare forward-pass spikes; NaN detection is the last
    line of defence.

    `model.make_neighbor_hooks()` is supplied by both AIMNet2Wrapper and
    PipelineModelWrapper; they return the model-specific neighbour-list
    + periodic-image hooks needed for that architecture.
    """
    extra = ["stress"] if track_stress else []
    return [
        *model.make_neighbor_hooks(),
        WrapPeriodicHook(stage=DynamicsStage.AFTER_POST_UPDATE),
        MaxForceClampHook(max_force=max_force),
        NaNDetectorHook(extra_keys=extra),
    ]

### `LoggingHook` and its custom-scalar protocol

The `LoggingHook` records per-step quantities into one of two backends: **`backend="csv"`** writes a row per (step, graph) to disk, and **`backend="custom"`** invokes a user-supplied `writer_fn(step, rows)` callable (we use it to mirror progress to stdout in Jupyter). The default columns are `step`, `graph_idx`, `status`, `energy`, `fmax`, `temperature`.

Beyond defaults, we add three **custom scalars** — pressure, volume, density — each a function `(ctx) -> Tensor[num_graphs]` that returns one value per graph each time the hook fires. The dict is what we pass as `custom_scalars=...` to `LoggingHook`.

We also expose `P_1ATM` (1 atm in nvalchemi's pressure unit, eV/Å³), used by the NPT integrator below.

In [ ]:
# ─── LoggingHook custom-scalar protocol ────────────────────────────────────
# Pressure unit conversion: nvalchemi expects pressure in eV/A^3 (matching
# the autograd-stress output). 1 atm = 101325 Pa; 1 eV/A^3 = 1.602e11 Pa.
P_1ATM = 101325.0 / 1.602176634e11  # eV/A^3


def stdout_writer(step, rows):
    """LoggingHook custom-backend writer: print one line per graph per step.

    `rows` is a list of dicts (one per graph), each containing the default
    columns (step, graph_idx, status, energy, fmax, temperature) plus any
    custom_scalars we configured."""
    for row in rows:
        parts = [
            f"{k}={v:.4g}" for k, v in row.items() if k not in ("graph_idx", "status")
        ]
        print(f"  [{int(step):>6d}] {' | '.join(parts)}")


def replay_log_csv(csv_path, writer=None):
    """Print a cached LoggingHook CSV through a stdout writer.

    Reads the CSV row-by-row, groups rows by step, calls writer(step, rows)
    once per step. The writer receives a list of dicts (one per graph) — the
    same contract a live LoggingHook custom-backend writer satisfies. Pass
    writer=stdout_writer for single-graph stages or
    make_graph_tagged_writer(labels) for multi-graph (defined in §8).
    """
    writer = writer or stdout_writer
    rows_by_step = {}
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        for raw in reader:
            step = int(float(raw["step"]))
            row = {}
            for k, v in raw.items():
                if k == "step":
                    continue
                if k == "graph_idx":
                    row[k] = int(float(v))
                else:
                    try:
                        row[k] = float(v)
                    except (ValueError, TypeError):
                        row[k] = v
            rows_by_step.setdefault(step, []).append(row)
    for step in sorted(rows_by_step):
        writer(step, rows_by_step[step])


# ── Each function receives the full hook ctx and must return a
# Tensor[num_graphs] — one scalar per graph per step.
def pressure_scalar(ctx):
    """Instantaneous scalar pressure per graph (eV/A^3).
    P = Tr(stress)/3 + 2 KE / (3 V), nvalchemi stress is compression-positive."""
    batch = ctx.batch
    stress_trace = batch.stress.diagonal(dim1=-2, dim2=-1).mean(dim=-1)
    V = torch.linalg.det(batch.cell).abs().view(-1)
    ke_per_atom = 0.5 * batch.atomic_masses * (batch.velocities**2).sum(dim=-1)
    ke_per_graph = torch.zeros(batch.num_graphs, device=V.device, dtype=V.dtype)
    ke_per_graph.scatter_add_(0, batch.batch_idx, ke_per_atom.to(V.dtype))
    return stress_trace + (2.0 / 3.0) * ke_per_graph / V


def volume_scalar(ctx):
    """Cell volume per graph (A^3)."""
    return torch.linalg.det(ctx.batch.cell).abs().view(-1)


def density_scalar(ctx):
    """Density per graph (g/cm^3) — same math as compute_density above."""
    vol = torch.linalg.det(ctx.batch.cell).abs().view(-1)
    mass = torch.zeros(ctx.batch.num_graphs, device=vol.device)
    mass.scatter_add_(0, ctx.batch.batch_idx, ctx.batch.atomic_masses)
    return mass * AMU_OVER_A3_TO_G_CM3 / vol


# Pass DYNAMICS_SCALARS directly to LoggingHook(custom_scalars=...) to add
# pressure / volume / density columns alongside the default ones.
DYNAMICS_SCALARS = {
    "pressure_eV_A3": pressure_scalar,
    "volume_A3": volume_scalar,
    "density_g_cm3": density_scalar,
}

### Plumbing helpers we'll import

The remaining pieces are pure file-IO conventions — checkpointing, Zarr trajectory sinks, JSON metadata for resume bookkeeping — and are not pertinent to the toolkit's dynamics behaviour. We import them rather than redefine them:

| Helper | Purpose |
|--------|---------|
| `checkpoint_exists(name, log_dir)` | Has stage `<name>` already written its end-of-stage Batch checkpoint? |
| `save_checkpoint(batch, name, log_dir)` / `load_checkpoint(...)` | Persist / restore a single end-of-stage `Batch` to `checkpoints/after_<name>.zarr`. |
| `save_stage_meta(name, log_dir, n_steps)` / `load_stage_meta(...)` | JSON metadata (`{"steps_completed": ...}`) so a partial run can extend rather than restart. |
| `integrator_state_exists` / `save_integrator_state` / `load_integrator_state` | Persist the NPT integrator's `_state` (NHC chains, barostat momenta) for transient-free resume. |
| `next_part_index` / `part_paths` | Multi-part log filenames (`stem.csv`, `stem.part2.csv`, ...) so an extension run doesn't clobber the original. |
| `fresh_zarr_sink(path, capacity)` | Construct a `ZarrData` sink with a pre-sized buffer. |

In [ ]:
# ─── Plumbing helpers (file naming, checkpointing, Zarr sinks) ─────────────
from helpers import (
    checkpoint_exists,
    fresh_zarr_sink,
    integrator_state_exists,
    load_checkpoint,
    load_integrator_state,
    load_stage_meta,
    next_part_index,
    part_paths,
    save_checkpoint,
    save_integrator_state,
    save_stage_meta,
)

In [ ]:
# ─── Warmup pipeline setup ─────────────────────────────────────────────────
# Each sub-stage writes its zarr trajectory + CSV log + checkpoint under
# logs/<RUN_NAME>/. On rerun, completed stages are skipped automatically by
# checking checkpoint_exists() / load_stage_meta(); partially completed
# stages extend from where they left off (.part2.zarr, .part2.csv).
#
# In FAST_DEMO mode all three sub-stages are skipped; the cached warmup-NPT
# extxyz is loaded directly for the diagnostics in §5.

# Integrators driving each warmup sub-stage:
#   FIRE2          — geometry minimiser (pseudo-dynamics dt, fmax termination)
#   NVTLangevin    — canonical-ensemble thermalisation
#   NPT            — anisotropic isobaric-isothermal cell relaxation
from nvalchemi.dynamics.optimizers.fire2 import FIRE2
from nvalchemi.dynamics.integrators.nvt_langevin import NVTLangevin
from nvalchemi.dynamics.integrators.npt import NPT

# Hooks fed into every integrator below:
#   LoggingHook    — per-step CSV / stdout writer (incl. DYNAMICS_SCALARS)
#   SnapshotHook   — per-step Zarr trajectory sink
# initialize_velocities seeds Boltzmann velocities at T_WARMUP before NVT.
# ConvergenceHook.from_fmax provides FIRE2's fmax termination criterion.
from nvalchemi.dynamics.hooks import LoggingHook, SnapshotHook
from nvalchemi.dynamics import initialize_velocities
from nvalchemi.dynamics.base import ConvergenceHook


def warmup_stage_names(t_warmup):
    """Stage name triple matching the warmup_naphthalene.py driver,
    so cached files cross-reference live-MD output verbatim."""
    tag = f"{int(t_warmup)}k"
    return ("fire", f"nvt_{tag}", f"npt_{tag}")


n_nvt = int(THERMALIZE_PS * 1000 / DT)
n_npt = int(EQUILIBRATE_PS * 1000 / DT)
fire_ck, nvt_ck, npt_ck = warmup_stage_names(T_WARMUP)
fire_fs = f"warmup_fire_{DT_TAG}"
nvt_fs = f"warmup_nvt_{T_WARMUP_TAG}_{DT_TAG}"
npt_fs = f"warmup_npt_{T_WARMUP_TAG}_{DT_TAG}"

print(f"Warmup stages: {fire_ck} -> {nvt_ck} -> {npt_ck}")

if FAST_DEMO:
    print(
        "\nFAST_DEMO=True: replaying cached logs — "
        "live cost on A100 ≈ 2 min FIRE / 15 min NVT / 50 min NPT."
    )
    _warmup_t0 = None
else:
    _warmup_t0 = time.monotonic()

### Stage A — FIRE2 minimisation

Drive forces below `FMAX` eV/Å. FIRE2 is pseudo-dynamics: an internal `dt` ≈ 0.01 (unrelated to the physical MD timestep) couples velocity to force with adaptive stepsize. We disable stress tracking in `make_safety_hooks` for this stage — FIRE2 doesn't propagate autograd through the stress tape. The `ConvergenceHook.from_fmax` factory wires the fmax termination criterion as a hook, the same composition mechanic introduced above.

In [ ]:
# ─── Stage A — FIRE2 minimisation ──────────────────────────────────────────
if not FAST_DEMO:
    if checkpoint_exists(fire_ck, LOG_DIR):
        logger.info("[FIRE] skip (checkpoint exists)")
        batch = load_checkpoint(fire_ck, LOG_DIR, DEVICE)
    else:
        logger.info("[FIRE] start (<={} steps)", FIRE_MAX_STEPS)
        fire_zarr = fresh_zarr_sink(
            LOG_DIR / f"{fire_fs}.zarr",
            capacity=FIRE_MAX_STEPS // SNAPSHOT_EVERY + 10,
        )
        fire_csv = LoggingHook(
            backend="csv",
            custom_scalars=DYNAMICS_SCALARS,
            log_path=str(LOG_DIR / f"{fire_fs}.csv"),
            frequency=LOG_EVERY,
        )
        fire_out = LoggingHook(
            backend="custom",
            writer_fn=stdout_writer,
            custom_scalars=DYNAMICS_SCALARS,
            frequency=LOG_EVERY,
        )
        fire_stage = FIRE2(
            model=aimnet2,
            dt=0.01,
            n_steps=FIRE_MAX_STEPS,
            convergence_hook=ConvergenceHook.from_fmax(threshold=FMAX),
        )
        for h in [
            *make_safety_hooks(aimnet2, track_stress=False),
            SnapshotHook(sink=fire_zarr, frequency=SNAPSHOT_EVERY),
            fire_csv,
            fire_out,
        ]:
            fire_stage.register_hook(h)
        with fire_csv, fire_out:
            batch = fire_stage.run(batch)
        save_checkpoint(batch, fire_ck, LOG_DIR)
        save_stage_meta(fire_ck, LOG_DIR, FIRE_MAX_STEPS)
else:
    log = cached_log_csv(fire_fs)
    if log is not None:
        replay_log_csv(log)


### Stage B — NVT thermalisation

Reseed velocities from a Maxwell–Boltzmann distribution at `T_WARMUP` (`initialize_velocities` handles per-graph mass weighting + COM removal + KE rescaling), then run Langevin NVT for `THERMALIZE_PS` ps. Langevin is *memoryless* — there is no integrator state to checkpoint — so the resume path here is simpler than NPT below.

In [ ]:
# ─── Stage B — NVT thermalisation @ T_WARMUP ───────────────────────────────
if not FAST_DEMO:
    nvt_meta = load_stage_meta(nvt_ck, LOG_DIR)
    nvt_done = int(nvt_meta["steps_completed"]) if nvt_meta else 0
    if checkpoint_exists(nvt_ck, LOG_DIR) and nvt_done >= n_nvt:
        logger.info("[NVT] skip (checkpoint covers {} >= {} steps)", nvt_done, n_nvt)
        batch = load_checkpoint(nvt_ck, LOG_DIR, DEVICE)
    else:
        n_delta = n_nvt - nvt_done if checkpoint_exists(nvt_ck, LOG_DIR) else n_nvt
        if checkpoint_exists(nvt_ck, LOG_DIR):
            logger.info("[NVT] extend (+{} steps)", n_delta)
            batch = load_checkpoint(nvt_ck, LOG_DIR, DEVICE)
            part = next_part_index(LOG_DIR, nvt_fs)
        else:
            logger.info("[NVT] start ({} steps)", n_delta)
            batch.velocities = torch.zeros_like(batch.positions)
            initialize_velocities(
                batch.velocities,
                batch.atomic_masses,
                temperature=torch.tensor([T_WARMUP], device=DEVICE),
                batch_idx=batch.batch_idx,
                random_seed=42,
                remove_com=True,
                rescale=True,
            )
            part = 1
        nvt_csv_path, nvt_zarr_path = part_paths(LOG_DIR, nvt_fs, part)
        nvt_zarr = fresh_zarr_sink(
            nvt_zarr_path, capacity=n_delta // SNAPSHOT_EVERY + 10
        )
        nvt_csv = LoggingHook(
            backend="csv",
            custom_scalars=DYNAMICS_SCALARS,
            log_path=str(nvt_csv_path),
            frequency=LOG_EVERY,
        )
        nvt_out = LoggingHook(
            backend="custom",
            writer_fn=stdout_writer,
            custom_scalars=DYNAMICS_SCALARS,
            frequency=LOG_EVERY,
        )
        nvt_stage = NVTLangevin(
            model=aimnet2,
            dt=DT,
            temperature=T_WARMUP,
            friction=FRICTION,
            n_steps=n_delta,
        )
        for h in [
            *make_safety_hooks(aimnet2),
            SnapshotHook(sink=nvt_zarr, frequency=SNAPSHOT_EVERY),
            nvt_csv,
            nvt_out,
        ]:
            nvt_stage.register_hook(h)
        with nvt_csv, nvt_out:
            batch = nvt_stage.run(batch)
        save_checkpoint(batch, nvt_ck, LOG_DIR)
        save_stage_meta(nvt_ck, LOG_DIR, n_nvt)
else:
    log = cached_log_csv(nvt_fs)
    if log is not None:
        replay_log_csv(log)


### Stage C — anisotropic NPT

Couple a Martyna–Tobias–Klein barostat (τ_P = `BAROSTAT_TIME_FS` fs) at 1 atm and a Nosé–Hoover chain thermostat (τ_T = `THERMOSTAT_TIME` fs) at `T_WARMUP`. The `pressure` argument is a `[1, 3]` tensor — one target value per diagonal cell axis, broadcast to every graph in the batch. The cell relaxes anisotropically over `EQUILIBRATE_PS` ps.

Unlike Langevin NVT, the NPT integrator carries internal state — the Nosé–Hoover chain and barostat momenta — that we persist alongside the configuration via `save_integrator_state` / `load_integrator_state`. **Pre-populating `npt_stage._state` before the first call to `run()`** lets the integrator pick up exactly where it left off (no thermostat-chain transients on resume); this is one of the toolkit's quieter but most useful conveniences.

In [ ]:
# ─── Stage C — anisotropic NPT @ T_WARMUP ──────────────────────────────────
if not FAST_DEMO:
    npt_meta = load_stage_meta(npt_ck, LOG_DIR)
    npt_done = int(npt_meta["steps_completed"]) if npt_meta else 0
    npt_can_extend = checkpoint_exists(npt_ck, LOG_DIR) and integrator_state_exists(
        npt_ck, LOG_DIR
    )
    if checkpoint_exists(npt_ck, LOG_DIR) and npt_done >= n_npt:
        logger.info("[NPT] skip (checkpoint covers {} >= {} steps)", npt_done, n_npt)
        batch = load_checkpoint(npt_ck, LOG_DIR, DEVICE)
    else:
        if npt_can_extend:
            n_delta = n_npt - npt_done
            logger.info("[NPT] extend (+{} steps)", n_delta)
            batch = load_checkpoint(npt_ck, LOG_DIR, DEVICE)
            preloaded_state = load_integrator_state(npt_ck, LOG_DIR, DEVICE)
            part = next_part_index(LOG_DIR, npt_fs)
        else:
            logger.info("[NPT] start ({} steps)", n_npt)
            n_delta = n_npt
            preloaded_state = None
            part = 1
        npt_csv_path, npt_zarr_path = part_paths(LOG_DIR, npt_fs, part)
        npt_zarr = fresh_zarr_sink(
            npt_zarr_path, capacity=n_delta // SNAPSHOT_EVERY + 10
        )
        npt_csv = LoggingHook(
            backend="csv",
            custom_scalars=DYNAMICS_SCALARS,
            log_path=str(npt_csv_path),
            frequency=LOG_EVERY,
        )
        npt_out = LoggingHook(
            backend="custom",
            writer_fn=stdout_writer,
            custom_scalars=DYNAMICS_SCALARS,
            frequency=LOG_EVERY,
        )
        npt_stage = NPT(
            model=aimnet2,
            dt=DT,
            temperature=T_WARMUP,
            pressure=torch.tensor([[P_1ATM, P_1ATM, P_1ATM]], dtype=torch.float32),
            barostat_time=BAROSTAT_TIME_FS,
            thermostat_time=THERMOSTAT_TIME,
            pressure_coupling="anisotropic",
            n_steps=n_delta,
        )
        for h in [
            *make_safety_hooks(aimnet2),
            SnapshotHook(sink=npt_zarr, frequency=SNAPSHOT_EVERY),
            npt_csv,
            npt_out,
        ]:
            npt_stage.register_hook(h)
        # Integrator state preload: the integrator's first call to
        # _ensure_state_initialized is a no-op when _state already exists,
        # so the saved NHC + barostat momenta are used verbatim — fully
        # transient-free resume.
        if preloaded_state is not None:
            npt_stage._state = preloaded_state
        with npt_csv, npt_out:
            batch = npt_stage.run(batch)
        save_checkpoint(batch, npt_ck, LOG_DIR)
        save_integrator_state(npt_stage._state, npt_ck, LOG_DIR)
        save_stage_meta(npt_ck, LOG_DIR, n_npt)
else:
    log = cached_log_csv(npt_fs)
    if log is not None:
        replay_log_csv(log)


In [ ]:
# ─── Warmup summary ────────────────────────────────────────────────────────
if FAST_DEMO:
    log = cached_log_csv(npt_fs)
    if log is not None:
        with open(log) as f:
            *_, last = csv.DictReader(f)
        print(
            f"\nWarmup done (replayed): fmax={float(last['fmax']):.4f} eV/A, "
            f"density={float(last['density_g_cm3']):.3f} g/cm^3"
        )
else:
    fmax_final = batch.forces.norm(dim=-1).max().item()
    print(
        f"\nWarmup done: fmax={fmax_final:.4f} eV/A, "
        f"density={compute_density(batch)[0]:.3f} g/cm^3, "
        f"elapsed={time.monotonic() - _warmup_t0:.1f}s"
    )

## 5. Warmup Diagnostics

Before cloning the warm crystal into the melt half (§6), we need quantitative confirmation that the warmup actually equilibrated. Three checks:

1. Density plateau in NPT.
2. COM-MSD plateau → diffusion coefficient $D \approx 0$.
3. Rotational order $S_0 \to 1$.

$D + S_0$ together constitute the phase classifier of Schmidt et al. [1] — applied here to the warmup endpoint, we want to see the **crystal** signature ($D \approx 0$, $S_0 \to 1$).

### Analysis helpers we'll use

Three of these are mathematically heavy enough that we import them rather than reproduce them — but it's worth knowing what each one does:

- **`compute_msd(snapshots, cells, n_atoms)`** — per-atom MSD with affine cell-deformation removed (per-frame fractional MSD using each frame's own cell). Returns `[n_frames-1, n_atoms]`.

- **`compute_com_msd(snapshots, cells, masses, atoms_per_mol)`** — the **physically meaningful** translational MSD. First reduces each molecule's atoms to a mass-weighted COM (PBC-unwrapped relative to atom 0), then subtracts the system COM (lab-frame raw atoms — see CLAUDE.md note on per-mol vs lab subtraction), then accumulates per-frame fractional drift. Returns `[n_frames-1, n_mol]`.

- **`compute_S0_from_frames(frames, atoms_per_mol)`** — chains principal-axis extraction → P₂ rotational ACF → tail average. Returns `(S0_mean, S0_per_axis, acf)`.

We **inline** the small numpy fit that converts COM-MSD → $D$ via the Einstein relation, so the conversion to cm²/s is fully visible.

In [ ]:
# ─── Analysis helpers: import the heavy math, inline the fit ───────────────
from helpers import (
    compute_S0_from_frames,
    compute_com_msd,
    compute_msd,
    load_warmup_trajectory,
    load_zarr_trajectory,
)


def fit_diffusion_coefficient(msd_per_mol, time_ps, fit_frac=0.5):
    """Einstein-relation D from the long-time slope of the mean COM MSD.

    Skips the ballistic / sub-diffusive head by fitting only the trailing
    `fit_frac` of the curve (the tail where MSD ~ 6 D t holds linearly).
    Unit conversion: 1 A^2/ps = 1e-4 cm^2/s.
    """
    msd_arr = np.asarray(msd_per_mol, dtype=np.float64)
    msd_mean = msd_arr.mean(axis=-1) if msd_arr.ndim == 2 else msd_arr
    time_ps = np.asarray(time_ps, dtype=np.float64)
    n_fit = max(2, int(msd_mean.shape[0] * fit_frac))
    fit_start_idx = msd_mean.shape[0] - n_fit
    slope, intercept = np.polyfit(time_ps[fit_start_idx:], msd_mean[fit_start_idx:], 1)
    d_A2_per_ps = float(slope) / 6.0  # 3D Einstein: MSD = 6 D t
    return {
        "D_A2_per_ps": d_A2_per_ps,
        "D_cm2_per_s": d_A2_per_ps * 1e-4,
        "slope": float(slope),
        "intercept": float(intercept),
        "fit_start_idx": int(fit_start_idx),
        "msd_mean": msd_mean,
    }

In [ ]:
# ─── Load warmup-NPT trajectory ────────────────────────────────────────────
# FAST_DEMO=True: read the cached extxyz and convert to Batch list.
# FAST_DEMO=False: load straight from the zarr just written by §4.
warmup_stem = f"warmup_npt_{T_WARMUP_TAG}_{DT_TAG}"
if FAST_DEMO:
    cached_path = cached_extxyz(warmup_stem)
    if cached_path is None:
        raise FileNotFoundError(
            f"FAST_DEMO=True but {cached_path} is missing. Run with "
            f"FAST_DEMO=False or stage the cached extxyz under {TRAJ_DIR}/."
        )
    print(f"Loading cached {cached_path}")
    npt_atoms = ase_read(str(cached_path), index=":")
    warmup_frames = ase_frames_to_batches(npt_atoms, device="cpu")
else:
    # In live mode we keep only the NPT slice (FIRE/NVT have very different
    # cell behaviour and would skew the per-frame density curve below).
    all_frames, all_labels = load_warmup_trajectory(
        LOG_DIR, device="cpu", t_warmup=T_WARMUP, stage_suffix=f"_{DT_TAG}"
    )
    _, _, npt_name = warmup_stage_names(T_WARMUP)
    warmup_frames = [b for b, lbl in zip(all_frames, all_labels) if lbl == npt_name]

snap_positions = [b.positions for b in warmup_frames]
snap_cells = [b.cell.squeeze() for b in warmup_frames]
print(
    f"NPT snapshots loaded: {len(warmup_frames)} frames, "
    f"{snap_positions[0].shape[0]} atoms each"
)
time_ps_npt = np.arange(len(warmup_frames)) * (SNAPSHOT_EVERY * DT) / 1000.0

In [ ]:
# ─── Energy / temperature / density / cell-length plots ────────────────────
# We derive density from compute_density() per-frame (no CSV needed),
# and cell lengths from batch.cell.squeeze().norm. This lets the
# FAST_DEMO path render without any access to the CSV log.
density_npt = np.array([compute_density(b)[0] for b in warmup_frames])
cell_lengths_npt = np.stack(
    [b.cell.squeeze().norm(dim=-1).cpu().numpy() for b in warmup_frames]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.plot(time_ps_npt, density_npt, color="#2e7d32", lw=1.4, label="Simulation")
ax.axhline(1.18, color="gray", ls="--", label="Experimental (298 K)")
ax.set_xlabel("Time (ps)")
ax.set_ylabel("Density (g/cm$^3$)")
ax.set_title("Density during warmup NPT (should plateau)")
ax.legend(loc="lower right")

ax = axes[1]
for i, label in enumerate(["|a|", "|b|", "|c|"]):
    ax.plot(time_ps_npt, cell_lengths_npt[:, i], lw=1.3, label=label)
ax.set_xlabel("Time (ps)")
ax.set_ylabel("Cell length (A)")
ax.set_title("Anisotropic cell-vector relaxation (NPT)")
ax.legend(loc="lower right")
fig.suptitle(f"Warmup NPT diagnostics — T = {T_WARMUP} K", fontsize=13)
plt.tight_layout()
plt.savefig(LOG_DIR / "warmup_diagnostics.png", dpi=120, bbox_inches="tight")
plt.show()

n_tail = max(1, len(density_npt) // 5)
print(
    f"Density (last 20% of NPT, n={n_tail}): "
    f"{density_npt[-n_tail:].mean():.3f} ± {density_npt[-n_tail:].std():.3f} g/cm^3"
)
print("Experimental (298 K):                 1.18 g/cm^3")

In [ ]:
# ─── COM-MSD + Einstein D fit ──────────────────────────────────────────────
# compute_com_msd handles affine cell-deformation removal AND lab-frame
# system-COM subtraction (both essential under anisotropic NPT — without
# them, an asymmetrically expanding cell injects 1-2 orders of magnitude
# of fictitious diffusion). Returns [n_frames-1, n_mol] cumulative MSD.
# We then fit the trailing 50% to the 3D Einstein relation MSD = 6 D t.
masses_ref = warmup_frames[0].atomic_masses
msd_per_mol = compute_com_msd(snap_positions, snap_cells, masses_ref, ATOMS_PER_MOL)
# msd_per_mol corresponds to time_ps_npt[1:] (compute_com_msd refs frame 0).
fit = fit_diffusion_coefficient(msd_per_mol.numpy(), time_ps_npt[1:], fit_frac=0.5)
D_cm2_per_s = fit["D_cm2_per_s"]
msd_mean = fit["msd_mean"]
fit_start = fit["fit_start_idx"]
fit_t = time_ps_npt[1:][fit_start:]
fit_line = fit["slope"] * fit_t + fit["intercept"]

if abs(D_cm2_per_s) < 1e-7:
    regime = "crystal (D indistinguishable from 0)"
elif abs(D_cm2_per_s) < 1e-6:
    regime = "plastic-crystal-like"
else:
    regime = "liquid-like"

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(
    time_ps_npt[1:],
    msd_mean,
    color="#37474f",
    lw=1.3,
    label="COM MSD (cross-molecule mean)",
)
ax.axvspan(
    fit_t[0], fit_t[-1], color="#cfd8dc", alpha=0.4, label="D fit window (last 50%)"
)
ax.plot(
    fit_t,
    fit_line,
    color="#c62828",
    ls="--",
    lw=1.6,
    label=f"Einstein fit  (D = {D_cm2_per_s:+.2e} cm$^2$/s)",
)
ax.set_xlabel("Time (ps)")
ax.set_ylabel("COM MSD (Å$^2$)")
ax.set_title(f"Translational diffusion @ T = {T_WARMUP} K  →  {regime}")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig(LOG_DIR / "warmup_msd.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"D = {D_cm2_per_s:+.3e} cm^2/s  (regime: {regime})")

In [ ]:
# ─── Rotational order parameter S0 ─────────────────────────────────────────
# P2 ACF of each molecule's three principal-inertia axes vs the first
# warmup-NPT frame; S_0 is the tail (last 20%) average. Crystal -> 1,
# plastic crystal -> 0..1, liquid -> 0.
S0_mean, S0_per_axis, acf = compute_S0_from_frames(
    warmup_frames,
    ATOMS_PER_MOL,
    ref_idx=0,
    tail_frac=0.2,
)

fig, ax = plt.subplots(figsize=(10, 4.5))
axis_labels = ["long axis (smallest I)", "short axis (middle I)", "normal (largest I)"]
axis_colors = ["#1565c0", "#2e7d32", "#c62828"]
for k, (lbl, col) in enumerate(zip(axis_labels, axis_colors)):
    ax.plot(time_ps_npt, acf[:, k], lw=1.3, color=col, label=lbl)
ax.plot(
    time_ps_npt,
    acf.mean(axis=1),
    lw=2.0,
    color="black",
    ls="--",
    label=f"mean S_0 = {S0_mean:+.3f}",
)
ax.axhline(1.0, color="gray", ls=":", lw=0.8)
ax.axhline(0.0, color="gray", ls=":", lw=0.8)
ax.set_ylim(-0.15, 1.15)
ax.set_xlabel("Time (ps)")
ax.set_ylabel("P$_2$ rotational ACF")
ax.set_title(f"S_0 per molecular axis (warmup NPT @ T = {T_WARMUP} K)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.savefig(LOG_DIR / "warmup_S0.png", dpi=120, bbox_inches="tight")
plt.show()

print(
    f"S_0 per axis (long/short/normal): {S0_per_axis[0]:+.3f}  "
    f"{S0_per_axis[1]:+.3f}  {S0_per_axis[2]:+.3f}"
)
print(f"S_0 mean:                         {S0_mean:+.3f}")

> **Phase read.** With $D \approx 0$ (well below the $10^{-7}$ cm²/s crystal threshold) and $S_0 \to 1$, the warmup endpoint is a crystal — exactly what we want as the solid input to SLC. A liquid-like read here would force us to either lower $T_\mathrm{warmup}$ or audit the model / equilibration setup before proceeding.

## 6. Generating the Liquid Half

We need an equilibrated liquid configuration with the same in-plane cell as the warm crystal so the two halves stack cleanly in §7. The recipe: clone the warmup-NPT endpoint, **reseed Maxwell–Boltzmann velocities at $T_\mathrm{melt} = $ `T_MELT` K** (well above $T_\mathrm{m,exp}$), then run NVT Langevin for `MELT_PS` ps. Why NVT and not NPT? We want the liquid's cell to match the solid's exactly — letting the barostat relax the box during melting would defeat the purpose. The fixed cell ends up slightly over-pressurised but the liquid relaxes in §9 once it shares a cell with the solid.

In [ ]:
# ─── Melt NVT @ T_MELT ─────────────────────────────────────────────────────
# All helpers we need (checkpoint_*, save_*, fresh_zarr_sink, compute_*) are
# already imported. NVTLangevin is stateless (the Langevin thermostat is
# memoryless), so the resume path is simpler than the NPT case in §4 — no
# integrator _state to save / load.
melt_stem = f"melt_nvt_{T_MELT_TAG}_from_{MELT_SRC}_{T_WARMUP_TAG}_{DT_TAG}"
melt_ck = f"meltgen_nvt_{T_MELT_TAG}_from_{MELT_SRC}_{T_WARMUP_TAG}_{DT_TAG}"

if FAST_DEMO:
    log = cached_log_csv(melt_stem)
    if log is not None:
        replay_log_csv(log)
    cached_path = cached_extxyz(melt_stem)
    if cached_path is None:
        raise FileNotFoundError(f"Missing {cached_path}; run with FAST_DEMO=False.")
    print(f"Loading cached {cached_path}")
    melt_atoms = ase_read(str(cached_path), index=":")
    melt_frames = ase_frames_to_batches(melt_atoms, device="cpu")
    melt_batch = melt_frames[-1]
else:
    n_melt = int(MELT_PS * 1000 / DT)
    melt_meta = load_stage_meta(melt_ck, LOG_DIR)
    melt_done = int(melt_meta["steps_completed"]) if melt_meta else 0
    has_ckpt = checkpoint_exists(melt_ck, LOG_DIR)

    if has_ckpt and melt_done >= n_melt:
        logger.info("[MELT] skip (checkpoint covers {} steps)", melt_done)
        melt_batch = load_checkpoint(melt_ck, LOG_DIR, DEVICE)
    else:
        if has_ckpt:
            n_delta = n_melt - melt_done
            logger.info("[MELT] extend (+{} steps)", n_delta)
            melt_batch = load_checkpoint(melt_ck, LOG_DIR, DEVICE)
            part = next_part_index(LOG_DIR, melt_stem)
        else:
            n_delta = n_melt
            logger.info("[MELT] start ({} steps)", n_delta)
            melt_batch = batch.clone()
            initialize_velocities(
                melt_batch.velocities,
                melt_batch.atomic_masses,
                temperature=torch.tensor([T_MELT], device=DEVICE),
                batch_idx=melt_batch.batch_idx,
                random_seed=123,
                remove_com=True,
                rescale=True,
            )
            part = 1
        melt_csv_path, melt_zarr_path = part_paths(LOG_DIR, melt_stem, part)
        melt_zarr = fresh_zarr_sink(
            melt_zarr_path, capacity=n_delta // SNAPSHOT_EVERY + 10
        )
        melt_csv = LoggingHook(
            backend="csv",
            custom_scalars=DYNAMICS_SCALARS,
            log_path=str(melt_csv_path),
            frequency=LOG_EVERY,
        )
        melt_out = LoggingHook(
            backend="custom",
            writer_fn=stdout_writer,
            custom_scalars=DYNAMICS_SCALARS,
            frequency=LOG_EVERY,
        )
        nvt_melt = NVTLangevin(
            model=aimnet2,
            dt=DT,
            temperature=T_MELT,
            friction=FRICTION,
            n_steps=n_delta,
        )
        for h in [
            *make_safety_hooks(aimnet2),
            SnapshotHook(sink=melt_zarr, frequency=SNAPSHOT_EVERY),
            melt_csv,
            melt_out,
        ]:
            nvt_melt.register_hook(h)
        with melt_csv, melt_out:
            melt_batch = nvt_melt.run(melt_batch)
        save_checkpoint(melt_batch, melt_ck, LOG_DIR)
        save_stage_meta(melt_ck, LOG_DIR, n_melt)
    melt_frames = load_zarr_trajectory(str(LOG_DIR / f"{melt_stem}.zarr"), device="cpu")

# Liquid signature: D >> 0, S_0 -> 0. We compute S_0 as a single number
# (no need to plot the ACF curve again — we just want the contrast vs
# the crystal).
melt_S0, _, _ = compute_S0_from_frames(
    melt_frames,
    ATOMS_PER_MOL,
    ref_idx=0,
    tail_frac=0.2,
)
print(f"Melt-NVT endpoint: S_0 = {melt_S0:+.3f}  (expect << crystal {S0_mean:+.3f})")

## 7. SLC Construction

Stack the warmed crystal above the liquid along the **monoclinic unique axis $b$**, doubling the cell's $b$-vector. Three subtle issues need handling:

**(a) Stacking axis.** For $P2_1/a$ naphthalene, only $b$ has the property $\vec b \parallel \vec b^*$ (direct lattice vector perpendicular to its Miller-plane face). Stacking along $a$ or $c$ gives an oblique interface that bends under anisotropic NPT.

**(b) Vacuum gap.** Naive `cat([crystal, melt + b_vec])` produces sub-Å clashes at the crystal–melt boundary (atoms at the edge of each half collide with their neighbour-image counterparts). We insert a 4 Å gap, comfortably below the AIMNet2 cutoff but above the protrusion of unwrapped molecules.

**(c) Per-half molecule unwrap.** Both halves arrive PBC-wrapped: molecules straddling the cell boundary have been mapped back into the box so their atoms appear on opposite faces. After concatenation + doubling, those split molecules persist as torn radicals with atoms separated by ~$|b|$. We unwrap each half in its own original cell first, then re-wrap by molecule centre-of-mass into the doubled cell.

### One imported helper

**`min_pbc_distance(positions, cell)`** is a sanity check we run after stacking: chunked O(N²) PBC-aware nearest-neighbour distance. We fail fast if it returns < 0.5 Å (atomic clash) — such a clash is a sign the gap was insufficient or one of the halves wasn't unwrapped properly. Pure utility, so we import.

The molecule-detection and unwrap functions, by contrast, are **defined inline** because they touch toolkit-relevant detail (Batch atom-ordering, PBC fractional unwrap) that the reader should see.

In [ ]:
# ─── Molecule-detection + unwrap helpers (defined inline) ─────────────────
# nvalchemi Batch atoms are species-grouped (200 × C10H8 -> [C × 2000,
# H × 1600]), so a naive 'walk 18 atoms at a time' would pick up the
# wrong atoms. Recovery from a torch.Tensor uses PBC-aware connectivity
# via ASE's primitive_neighbor_list with element-aware bond cutoffs.
# Same primitive that helpers/visualization.py uses; defined inline
# here because the SLC stack is the canonical place to learn the
# pattern.


def _detect_naphthalene_molecules(positions, atomic_numbers, cell):
    """Find naphthalene molecules in the Batch via element-pair bond cutoffs.

    Element-aware cutoffs: C-C 1.6 A, C-H 1.4 A; H-H pairs never bond.
    primitive_neighbor_list takes per-atom half-cutoffs whose pair-sum
    gives the bond length. Union-find collapses bond edges into 18-atom
    clusters — the per-molecule index lists.
    """
    from ase.neighborlist import primitive_neighbor_list

    pos_np = positions.detach().cpu().numpy()
    cell_np = cell.detach().cpu().numpy()
    Z = atomic_numbers.detach().cpu().numpy()
    cutoffs = np.where(Z == 6, 1.6 / 2, 1.4 / 2)  # half so pair-sum gives bond length
    i_idx, j_idx = primitive_neighbor_list(
        "ij", pbc=[True] * 3, cell=cell_np, positions=pos_np, cutoff=cutoffs
    )
    n = len(Z)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for ii, jj in zip(i_idx, j_idx):
        a, b = find(int(ii)), find(int(jj))
        if a != b:
            parent[a] = b
    groups = {}
    for k in range(n):
        groups.setdefault(find(k), []).append(k)
    return [sorted(g) for g in groups.values() if len(g) == 18]


def _unwrap_molecules(positions, cell, mol_indices):
    """MIC-unwrap each molecule's atoms relative to its first atom.

    Standard PBC fractional unwrap: dr_frac = (r - r0) @ inv(cell);
    dr_frac -= round(dr_frac); r_unwrapped = r0 + dr_frac @ cell.
    After this, every molecule is contiguous in real space (no atom is
    on the opposite face from its neighbour) and the doubled SLC cell
    can host the stack without torn-radical artefacts.
    """
    cell_inv = torch.linalg.inv(cell)
    out = positions.clone()
    for atoms in mol_indices:
        ref = out[atoms[0] : atoms[0] + 1]
        d = out[atoms] - ref
        df = d @ cell_inv
        df -= torch.round(df)
        out[atoms] = ref + df @ cell
    return out

### Building the SLC stack

With the molecule helpers in hand, build the stacked single-graph configuration:

1. Detect + unwrap molecules in **each half independently** (the crystal `Batch` from §4 and the melt `Batch` from §6, both PBC-wrapped in their original cells).
2. Translate the melt half by `|b| + GAP` along the b-axis unit vector.
3. Concatenate positions / atomic numbers, double the b-cell-vector, and rebuild a single-graph `Batch`.

The cell below also assembles the dynamics fields (`forces`, `energy`, `stress`, `velocities`) directly from torch tensors — `AtomicData.from_atoms` would require a round-trip through ASE that buys nothing here since we already have everything as device tensors. This is the alternative pattern to the `from_atoms` flow seen in §1 / §2: when the data is already in toolkit-native form, build `AtomicData` directly.

In [ ]:
# ─── SLC stack ─────────────────────────────────────────────────────────────
from helpers import min_pbc_distance

if FAST_DEMO:
    # The cached SLC initial configuration ships pre-built (with gap=5 A);
    # we load it directly so the geometry the rest of §8/§9 cells sees
    # matches what was actually run.
    init_stem = f"slc_init_from_{MELT_SRC}_{T_WARMUP_TAG}_{DT_TAG}_gap5p0A"
    cached_path = cached_extxyz(init_stem)
    if cached_path is None:
        raise FileNotFoundError(f"Missing {cached_path}; run with FAST_DEMO=False.")
    print(f"Loading cached {cached_path}")
    slc_init_atoms = ase_read(str(cached_path), index=0)
    n_slc = len(slc_init_atoms)
    slc_pos = torch.tensor(
        slc_init_atoms.get_positions(), dtype=torch.float32, device=DEVICE
    )
    slc_Z = torch.tensor(
        slc_init_atoms.get_atomic_numbers(), dtype=torch.long, device=DEVICE
    )
    slc_cell = torch.tensor(
        np.asarray(slc_init_atoms.cell), dtype=torch.float32, device=DEVICE
    )
    print(f"SLC system: {n_slc} atoms, cell |b|={slc_cell[1].norm():.2f} A")
else:
    crystal_batch = batch.clone()
    GAP_A = 4.0  # vacuum gap thickness along b (per face -> total 2 * GAP)
    cryst_mols = _detect_naphthalene_molecules(
        crystal_batch.positions,
        crystal_batch.atomic_numbers,
        crystal_batch.cell.squeeze(),
    )
    melt_mols = _detect_naphthalene_molecules(
        melt_batch.positions, melt_batch.atomic_numbers, melt_batch.cell.squeeze()
    )
    cryst_pos = _unwrap_molecules(
        crystal_batch.positions, crystal_batch.cell.squeeze(), cryst_mols
    )
    melt_pos = _unwrap_molecules(
        melt_batch.positions, melt_batch.cell.squeeze(), melt_mols
    )

    cell_orig = crystal_batch.cell.squeeze()
    b_vec = cell_orig[1, :]
    b_unit = b_vec / b_vec.norm()
    melt_pos = melt_pos + b_unit * (b_vec.norm() + GAP_A)
    slc_pos = torch.cat([cryst_pos, melt_pos], dim=0)
    slc_Z = torch.cat([crystal_batch.atomic_numbers, melt_batch.atomic_numbers], dim=0)
    n_slc = slc_pos.shape[0]
    slc_cell = cell_orig.clone()
    slc_cell[1, :] = b_unit * (2 * b_vec.norm() + 2 * GAP_A)

# Build the single-graph SLC Batch directly from device tensors (we have
# everything in hand already; no ASE detour). atomic_masses auto-populates
# from atomic_numbers via an AtomicData validator.
slc_data = AtomicData(
    positions=slc_pos,
    atomic_numbers=slc_Z,
    velocities=torch.zeros_like(slc_pos),
    forces=torch.zeros(n_slc, 3, device=DEVICE),
    energy=torch.zeros(1, 1, device=DEVICE),
    stress=torch.zeros(1, 3, 3, device=DEVICE),
    cell=slc_cell.unsqueeze(0),
    pbc=torch.tensor([[True, True, True]], device=DEVICE),
)
slc_data.charge = torch.zeros(1, 1, device=DEVICE)
slc_batch = Batch.from_data_list([slc_data], device=DEVICE)
n_half = slc_batch.num_nodes // 2
min_dist = min_pbc_distance(slc_batch.positions, slc_batch.cell.squeeze())
print(f"SLC: {slc_batch.num_nodes} atoms ({n_half} crystal + {n_half} melt)")
print(
    f"Cell: {[f'{length:.2f}' for length in slc_batch.cell.squeeze().norm(dim=-1).tolist()]} A"
)
print(f"Min PBC distance: {min_dist:.2f} A  (must be > 0.5 A)")

visualize_structure(
    slc_batch,
    title=f"SLC initial structure ({slc_batch.num_nodes} atoms)",
    save_path=str(LOG_DIR / "slc_construction.png"),
)


## 8. SLC Pre-equilibration

Following Schmidt et al. [1], we insert two short stages between SLC construction and the production NPT, in order:

1. **FIRE2 minimisation** on the stacked single-graph geometry. Drains residual elastic strain at the crystal–melt interface.
2. **Short NVT** (~10 ps) at each target temperature in `TEMPS`. Lets the interface relax thermally before the cell starts moving under barostat control. We run this as a **multi-graph batch** — one copy of the post-FIRE geometry per temperature — so all temperatures equilibrate in parallel.

Skipping pre-equilibration tends to produce visible barostat transients in the first ~50 ps of the production NPT and occasionally crashes from the multi-tau_P relaxation kick. The consolidated drivers (`slc_naphthalene.py`) always run this pre-equilibration sequence; this notebook mirrors them.

### A multi-graph LoggingHook writer

When five SLC systems run as one `Batch`, the default `stdout_writer` would emit five identical-looking lines per step. **`make_graph_tagged_writer(labels)`** wraps it into a closure that prefixes every row with `labels[graph_idx]` — e.g. `T=300K | step=... | energy=...`. The same `LoggingHook` infrastructure handles single-graph and multi-graph batches; the only thing that changes is the writer.

In [ ]:
# ─── Multi-graph stdout writer (defined inline) ────────────────────────────
def make_graph_tagged_writer(labels):
    """Stdout writer that prefixes each row with labels[graph_idx]."""

    def writer(step, rows):
        for row in rows:
            gi = int(row.get("graph_idx", 0))
            tag = labels[gi] if gi < len(labels) else f"g{gi}"
            parts = [
                f"{k}={v:.4g}"
                for k, v in row.items()
                if k not in ("graph_idx", "status")
            ]
            print(f"  [{int(step):>6d}] {tag} | {' | '.join(parts)}")

    return writer

In [ ]:
# ─── SLC pre-equilibration setup ───────────────────────────────────────────
# Two stages bridge construction (§7) and production NPT (§9):
#   (1) FIRE2 on the single-graph stack — drains residual elastic strain.
#   (2) Multi-graph NVT @ each TEMPS value — interface relaxes thermally
#       before barostat takes over. We run all temperatures in parallel as
#       one Batch with a per-graph temperature tensor.
# In FAST_DEMO mode both are skipped; the cached production NPT trajectories
# already reflect their effect.

n_slc_nvt = int(EQUIL_SLC_PS * 1000 / DT)
temps_tensor = torch.tensor([float(T) for T in TEMPS], device=DEVICE)
t_labels = [f"T={T}K" for T in TEMPS]
slc_fire_ck = f"slc_fire_from_{MELT_SRC}_{T_WARMUP_TAG}_{DT_TAG}"
slc_nvt_ck = f"slc_nvt_from_{MELT_SRC}_{T_WARMUP_TAG}_{DT_TAG}"
slc_fire_fs = slc_fire_ck
slc_nvt_fs = slc_nvt_ck

if FAST_DEMO:
    print(
        "FAST_DEMO=True: replaying cached logs — "
        "live cost on A100 ≈ 3 min FIRE / 15 min NVT / 3-6 hr NPT (multi-T sweep dominates)."
    )
    slc_multi_batch = None

### SLC FIRE2 minimisation

Same FIRE2 integrator as §4-Stage A, but on the stacked single-graph geometry. Convergence often comes well before `FIRE_MAX_STEPS`; we save the converged batch as `slc_fire_<...>` so subsequent runs short-circuit.

In [ ]:
# ─── SLC FIRE2 minimisation ────────────────────────────────────────────────
if not FAST_DEMO:
    if checkpoint_exists(slc_fire_ck, LOG_DIR):
        logger.info("[SLC FIRE] skip (checkpoint exists)")
        slc_batch = load_checkpoint(slc_fire_ck, LOG_DIR, DEVICE)
    else:
        logger.info("[SLC FIRE] start (<={} steps)", FIRE_MAX_STEPS)
        fire_zarr = fresh_zarr_sink(
            LOG_DIR / f"{slc_fire_fs}.zarr",
            capacity=FIRE_MAX_STEPS // SNAPSHOT_EVERY + 10,
        )
        fire_csv = LoggingHook(
            backend="csv",
            custom_scalars=DYNAMICS_SCALARS,
            log_path=str(LOG_DIR / f"{slc_fire_fs}.csv"),
            frequency=LOG_EVERY,
        )
        fire_out = LoggingHook(
            backend="custom",
            writer_fn=stdout_writer,
            custom_scalars=DYNAMICS_SCALARS,
            frequency=LOG_EVERY,
        )
        fire_stage = FIRE2(
            model=aimnet2,
            dt=0.01,
            n_steps=FIRE_MAX_STEPS,
            convergence_hook=ConvergenceHook.from_fmax(threshold=FMAX),
        )
        for h in [
            *make_safety_hooks(aimnet2, track_stress=False),
            SnapshotHook(sink=fire_zarr, frequency=SNAPSHOT_EVERY),
            fire_csv,
            fire_out,
        ]:
            fire_stage.register_hook(h)
        with fire_csv, fire_out:
            slc_batch = fire_stage.run(slc_batch)
        save_checkpoint(slc_batch, slc_fire_ck, LOG_DIR)
        save_stage_meta(slc_fire_ck, LOG_DIR, FIRE_MAX_STEPS)
else:
    log = cached_log_csv(slc_fire_fs)
    if log is not None:
        replay_log_csv(log)
    else:
        print("Cached SLC FIRE2 minimisation: log not in cache; resuming from cached endpoint.")


### Multi-graph NVT thermalisation per target temperature

Clone the FIRE-minimised single-graph geometry into one copy per temperature, seed velocities at each `TEMPS[i]`, thermalise in parallel for `EQUIL_SLC_PS` ps. Two patterns to notice:

- **Per-graph integrator parameters.** `NVTLangevin.temperature` accepts a `torch.Tensor[M]` and broadcasts via `batch.batch_idx`. The same broadcasting convention applies to `NPT.temperature` and `NPT.pressure` in §9 — once you build the multi-graph `Batch`, every per-graph parameter just works.
- **Multi-graph stdout writer.** `make_graph_tagged_writer(t_labels)` from above gives every progress line a `T=<value>K` prefix so multi-graph stdout output stays legible.

We construct each `AtomicData` directly from torch tensors (the FIRE2 output is already on `DEVICE`, so an ASE round-trip would be pure overhead).

In [ ]:
# ─── Multi-graph NVT @ per-T ───────────────────────────────────────────────
if not FAST_DEMO:
    nvt_meta = load_stage_meta(slc_nvt_ck, LOG_DIR)
    nvt_done = int(nvt_meta["steps_completed"]) if nvt_meta else 0
    if checkpoint_exists(slc_nvt_ck, LOG_DIR) and nvt_done >= n_slc_nvt:
        logger.info("[SLC NVT] skip")
        slc_multi_batch = load_checkpoint(slc_nvt_ck, LOG_DIR, DEVICE)
    else:
        slc_data_list = [
            AtomicData(
                positions=slc_batch.positions.clone(),
                atomic_numbers=slc_batch.atomic_numbers.clone(),
                velocities=torch.zeros_like(slc_batch.positions),
                forces=torch.zeros(n_slc, 3, device=DEVICE),
                energy=torch.zeros(1, 1, device=DEVICE),
                stress=torch.zeros(1, 3, 3, device=DEVICE),
                cell=slc_batch.cell.squeeze().clone().unsqueeze(0),
                pbc=torch.tensor([[True, True, True]], device=DEVICE),
            )
            for _ in TEMPS
        ]
        for d in slc_data_list:
            d.charge = torch.zeros(1, 1, device=DEVICE)
        slc_multi_batch = Batch.from_data_list(slc_data_list, device=DEVICE)
        initialize_velocities(
            slc_multi_batch.velocities,
            slc_multi_batch.atomic_masses,
            temperature=temps_tensor,
            batch_idx=slc_multi_batch.batch_idx,
            random_seed=42,
            remove_com=True,
            rescale=True,
        )
        nvt_zarr = fresh_zarr_sink(
            LOG_DIR / f"{slc_nvt_fs}.zarr",
            capacity=len(TEMPS) * (n_slc_nvt // SNAPSHOT_EVERY) + 10,
        )
        nvt_csv = LoggingHook(
            backend="csv",
            custom_scalars=DYNAMICS_SCALARS,
            log_path=str(LOG_DIR / f"{slc_nvt_fs}.csv"),
            frequency=LOG_EVERY,
        )
        nvt_out = LoggingHook(
            backend="custom",
            writer_fn=make_graph_tagged_writer(t_labels),
            custom_scalars=DYNAMICS_SCALARS,
            frequency=LOG_EVERY,
        )
        nvt_stage = NVTLangevin(
            model=aimnet2,
            dt=DT,
            temperature=temps_tensor,
            friction=FRICTION,
            n_steps=n_slc_nvt,
        )
        for h in [
            *make_safety_hooks(aimnet2),
            SnapshotHook(sink=nvt_zarr, frequency=SNAPSHOT_EVERY),
            nvt_csv,
            nvt_out,
        ]:
            nvt_stage.register_hook(h)
        with nvt_csv, nvt_out:
            slc_multi_batch = nvt_stage.run(slc_multi_batch)
        save_checkpoint(slc_multi_batch, slc_nvt_ck, LOG_DIR)
        save_stage_meta(slc_nvt_ck, LOG_DIR, n_slc_nvt)
    print(
        f"SLC pre-equilibration done; {slc_multi_batch.num_graphs} graphs ready for NPT."
    )
else:
    log = cached_log_csv(slc_nvt_fs)
    if log is not None:
        replay_log_csv(log, writer=make_graph_tagged_writer(t_labels))


## 9. SLC Production NPT

The production stage — anisotropic NPT at each target temperature for `SLC_PS` ps. The barostat acts independently per graph and per axis, so each temperature finds its own cell while the interface tracks coexistence. We run all five temperatures as one multi-graph `Batch`; per-temperature outputs are tagged via `graph_idx` in the CSV log and interleaved frame-by-frame in the Zarr trajectory.

> **Performance note.** Five temperatures × 200 ps × 0.5 fs = $2 \times 10^6$ steps × 5 graphs ≈ 6 GPU-hours on an A100. For production work, the `slc_multi_gpu.sh` wrapper splits the `TEMPS` list across multiple GPUs (`--temps 250,300` on rank 0, `--temps 350` on rank 1, etc.) for near-linear scaling.

### One imported helper for trajectory de-interleaving

**`extract_per_graph_trajectory(batches, graph_idx, num_graphs)`** is a thin slicing utility: `batches[graph_idx::num_graphs]`. `SnapshotHook` writes each graph as a separate Zarr sample, and `load_zarr_trajectory` returns them interleaved frame-by-frame (`[g0_s0, g1_s0, ..., g0_s1, g1_s1, ...]`). This helper turns that flat list back into a per-graph trajectory.

In [ ]:
# ─── SLC anisotropic NPT @ TEMPS ───────────────────────────────────────────
from helpers import extract_per_graph_trajectory

slc_npt_ck = f"slc_npt_from_{MELT_SRC}_{T_WARMUP_TAG}_{DT_TAG}"
slc_npt_fs = f"slc_all_from_{MELT_SRC}_{T_WARMUP_TAG}_{DT_TAG}"

if FAST_DEMO:
    log = cached_log_csv(slc_npt_fs)
    if log is not None:
        replay_log_csv(log, writer=make_graph_tagged_writer(t_labels))
    # Load each per-T extxyz separately (the cached files come from the
    # multi-GPU shell wrapper and are split into per-T traces). Build a
    # parallel structure mirroring extract_per_graph_trajectory's output:
    # results_per_T[T] = list[Batch] of frames for that temperature.
    results_per_T = {}
    for T in TEMPS:
        stem_T = f"{slc_npt_fs}_t{int(T)}"
        path = cached_extxyz(stem_T)
        if path is None:
            raise FileNotFoundError(f"Missing {path}; rerun multi-GPU SLC.")
        atoms_list = ase_read(str(path), index=":")
        results_per_T[T] = ase_frames_to_batches(atoms_list, device="cpu")
        print(f"  T={int(T)} K  →  {len(atoms_list)} frames")
else:
    n_steps_slc = int(SLC_PS * 1000 / DT)
    slc_meta = load_stage_meta(slc_npt_ck, LOG_DIR)
    slc_done = int(slc_meta["steps_completed"]) if slc_meta else 0
    can_extend = checkpoint_exists(slc_npt_ck, LOG_DIR) and integrator_state_exists(
        slc_npt_ck, LOG_DIR
    )

    if checkpoint_exists(slc_npt_ck, LOG_DIR) and slc_done >= n_steps_slc:
        logger.info("[SLC NPT] skip")
        final_batch = load_checkpoint(slc_npt_ck, LOG_DIR, DEVICE)
    else:
        if can_extend:
            n_delta = n_steps_slc - slc_done
            logger.info("[SLC NPT] extend (+{} steps)", n_delta)
            slc_multi_batch = load_checkpoint(slc_npt_ck, LOG_DIR, DEVICE)
            preloaded_state = load_integrator_state(slc_npt_ck, LOG_DIR, DEVICE)
            part = next_part_index(LOG_DIR, slc_npt_fs)
        else:
            n_delta = n_steps_slc
            preloaded_state = None
            part = 1

        slc_csv_path, slc_zarr_path = part_paths(LOG_DIR, slc_npt_fs, part)
        slc_zarr = fresh_zarr_sink(
            slc_zarr_path,
            capacity=len(TEMPS) * (n_delta // SNAPSHOT_EVERY) + 10,
        )
        t_labels = [f"T={T}K" for T in TEMPS]
        slc_csv = LoggingHook(
            backend="csv",
            custom_scalars=DYNAMICS_SCALARS,
            log_path=str(slc_csv_path),
            frequency=LOG_EVERY,
        )
        slc_out = LoggingHook(
            backend="custom",
            writer_fn=make_graph_tagged_writer(t_labels),
            custom_scalars=DYNAMICS_SCALARS,
            frequency=LOG_EVERY,
        )
        npt_slc = NPT(
            model=aimnet2,
            dt=DT,
            temperature=temps_tensor,
            pressure=torch.tensor([[P_1ATM, P_1ATM, P_1ATM]], dtype=torch.float32),
            barostat_time=BAROSTAT_TIME_FS,
            thermostat_time=THERMOSTAT_TIME,
            pressure_coupling="anisotropic",
            n_steps=n_delta,
        )
        for h in [
            *make_safety_hooks(aimnet2),
            SnapshotHook(sink=slc_zarr, frequency=SNAPSHOT_EVERY),
            slc_csv,
            slc_out,
        ]:
            npt_slc.register_hook(h)
        if preloaded_state is not None:
            npt_slc._state = preloaded_state
        with slc_csv, slc_out:
            final_batch = npt_slc.run(slc_multi_batch)
        save_checkpoint(final_batch, slc_npt_ck, LOG_DIR)
        save_integrator_state(npt_slc._state, slc_npt_ck, LOG_DIR)
        save_stage_meta(slc_npt_ck, LOG_DIR, n_steps_slc)

    # Reassemble per-T frame lists from the interleaved zarr.
    all_frames = load_zarr_trajectory(str(LOG_DIR / f"{slc_npt_fs}.zarr"), device="cpu")
    results_per_T = {
        T: extract_per_graph_trajectory(all_frames, i, len(TEMPS))
        for i, T in enumerate(TEMPS)
    }

print(f"\nProduction NPT loaded for {len(results_per_T)} temperatures")

## 10. $T_\mathrm{m}$ Extraction

Apply the phase classifier per temperature, separately to each half of the SLC system (`atom_slice=slice(0, n_half)` for the crystal half; `slice(n_half, None)` for the melt half). Three things should track $T_\mathrm{m}$:

**Crystal half $S_0$**: stays $\to 1$ below $T_\mathrm{m}$; drops toward 0 above $T_\mathrm{m}$ as the solid melts.

**Crystal-half COM-MSD**: plateaus below $T_\mathrm{m}$; grows linearly above.

**Density**: rises below $T_\mathrm{m}$ (more solid, less liquid); drops above (more liquid, lower density). The crossover sits at $T_\mathrm{m}$.

Three independent signatures pinning the same temperature is the confirmation pattern of Schmidt et al. [1].

In [ ]:
# ─── Per-T density / energy / S0 / MSD ─────────────────────────────────────
# All needed helpers (compute_S0_from_frames, compute_msd, compute_density)
# were imported / defined in §5 + §2. We just iterate over results_per_T.
n_half = list(results_per_T.values())[0][0].num_nodes // 2

per_T = {}
for T in TEMPS:
    frames_T = results_per_T[T]
    snap_pos = [b.positions for b in frames_T]
    snap_cell = [b.cell.squeeze() for b in frames_T]
    # Per-half S_0
    S0_c, _, _ = compute_S0_from_frames(
        frames_T,
        ATOMS_PER_MOL,
        ref_idx=0,
        tail_frac=0.2,
        atom_slice=slice(0, n_half),
    )
    S0_m, _, _ = compute_S0_from_frames(
        frames_T,
        ATOMS_PER_MOL,
        ref_idx=0,
        tail_frac=0.2,
        atom_slice=slice(n_half, None),
    )
    # Per-half COM-MSD (cumulative; final value is what we care about for melt)
    msd_full = compute_msd(snap_pos, snap_cell, frames_T[0].num_nodes)
    msd_c = msd_full[:, :n_half].mean(dim=1).numpy()
    msd_m = msd_full[:, n_half:].mean(dim=1).numpy()
    rho = np.array([compute_density(b)[0] for b in frames_T])
    per_T[T] = {
        "rho_mean": float(rho[len(rho) // 2 :].mean()),  # last 50% mean
        "msd_c_final": float(msd_c[-1]) if len(msd_c) else 0.0,
        "msd_m_final": float(msd_m[-1]) if len(msd_m) else 0.0,
        "S0_c": S0_c,
        "S0_m": S0_m,
    }

print(
    f"{'T (K)':>8} {'rho':>8} {'MSD_cryst':>10} {'MSD_melt':>10} {'S0_cryst':>10} {'S0_melt':>10}"
)
print("-" * 64)
for T in TEMPS:
    a = per_T[T]
    print(
        f"{int(T):>8} {a['rho_mean']:>8.3f} {a['msd_c_final']:>10.2f} "
        f"{a['msd_m_final']:>10.2f} {a['S0_c']:>10.3f} {a['S0_m']:>10.3f}"
    )

In [ ]:
# ─── Endpoint summary plot ─────────────────────────────────────────────────
T_arr = np.array(TEMPS)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

ax = axes[0]
ax.plot(
    T_arr, [per_T[T]["rho_mean"] for T in TEMPS], "o-", color="#2e7d32", markersize=8
)
ax.axvline(TM_EXP, color="gray", ls="--", label=f"T_m,exp = {TM_EXP} K")
ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Density (g/cm$^3$)")
ax.set_title("Final density vs T")
ax.legend()

ax = axes[1]
ax.plot(
    T_arr,
    [per_T[T]["msd_c_final"] for T in TEMPS],
    "s-",
    color="#1565c0",
    label="Crystal half",
)
ax.plot(
    T_arr,
    [per_T[T]["msd_m_final"] for T in TEMPS],
    "o-",
    color="#c62828",
    label="Melt half",
)
ax.axvline(TM_EXP, color="gray", ls="--")
ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Final MSD (Å$^2$)")
ax.set_title("Final MSD per half (crystal diverges above T_m)")
ax.legend()

ax = axes[2]
ax.plot(
    T_arr,
    [per_T[T]["S0_c"] for T in TEMPS],
    "s-",
    color="#1565c0",
    label="Crystal half",
)
ax.plot(
    T_arr, [per_T[T]["S0_m"] for T in TEMPS], "o-", color="#c62828", label="Melt half"
)
ax.axvline(TM_EXP, color="gray", ls="--")
ax.set_xlabel("Temperature (K)")
ax.set_ylabel("$S_0$")
ax.set_title("Rotational order per half")
ax.legend()

fig.suptitle(f"SLC endpoint summary — naphthalene  (T_m,exp = {TM_EXP} K)", fontsize=13)
plt.tight_layout()
plt.savefig(LOG_DIR / "slc_endpoints.png", dpi=120, bbox_inches="tight")
plt.show()

# Identify the bracket that contains T_m: lowest T at which crystal-half
# S0 has clearly dropped, or first T at which crystal MSD diverges.
S0_c_arr = np.array([per_T[T]["S0_c"] for T in TEMPS])
msd_c_arr = np.array([per_T[T]["msd_c_final"] for T in TEMPS])
S0_drop_idx = np.where(S0_c_arr < 0.5)[0]
msd_jump_idx = np.where(msd_c_arr > 5.0)[0]
if len(S0_drop_idx) > 0:
    bracket = (
        TEMPS[S0_drop_idx[0] - 1] if S0_drop_idx[0] > 0 else None,
        TEMPS[S0_drop_idx[0]],
    )
    print(f"\nT_m bracket from S_0 (cryst): {bracket}")
if len(msd_jump_idx) > 0:
    bracket = (
        TEMPS[msd_jump_idx[0] - 1] if msd_jump_idx[0] > 0 else None,
        TEMPS[msd_jump_idx[0]],
    )
    print(f"T_m bracket from MSD (cryst): {bracket}")
print(f"Reference: T_m,exp = {TM_EXP} K")

## Discussion

### Key observations

1. **The pipeline runs end-to-end on a single GPU**, with a single `AIMNet2Wrapper.from_checkpoint` call as the only model load. No force-field parameters were fit; no system-specific tuning beyond the supercell and target temperatures was needed.
2. **The Yoneya–Harada classifier is decisive.** $D$ alone or $S_0$ alone can be ambiguous (a plastic crystal has $D \approx 0$ and $0 < S_0 < 1$); together they pin the phase unambiguously and let us read $T_\mathrm{m}$ off the SLC sweep.
3. **Anisotropic NPT is mandatory**, not just preferable, for SLC. An isotropic barostat couples the in-plane axes (pinned by the solid lattice) to the interface-normal axis (which must move as the phase fraction shifts), biasing $T_\mathrm{m}$ by tens of kelvins.

### Caveats

- **AIMNet2-2025 cutoff vs. long-range interactions.** Naphthalene is a weak-dipole molecule, so the bare 5 Å cutoff is workable. For polar molecular crystals (paracetamol, glycine), wrap the model in `EwaldModelWrapper` to recover long-range electrostatics; pair it with `PipelineModelWrapper(use_autograd=True)` so forces and stress are differentiated through the summed (base + Ewald) energy.
- **`torch.compile`.** Use `compile_model=True` on `AIMNet2Wrapper.from_checkpoint(...)` to compile the inner forward pass; do **not** wrap `dynamics.run` itself in `torch.compile` — upstream graph breaks crash Inductor.
- **Finite-size effects.** A 200-molecule supercell is on the small side. Production $T_\mathrm{m}$ predictions need ≥ 1000 molecules and finer temperature sweeps (Schmidt et al. [1] use ≥ 1000 molecules per compound).
- **Single composition.** This notebook validates the procedure on one compound; benchmarking against experimental $T_\mathrm{m}$ needs a chemistry-spanning test set.

## You now know how to

- **Load a CIF and convert it to a toolkit `Batch`** via `AtomicData.from_atoms` plus `Batch.from_data_list`, with the dynamics fields (`forces`, `energy`, `stress`, `velocities`) explicitly pre-allocated — the canonical pattern for any ASE → toolkit conversion.
- **Compose a multi-stage MD pipeline** (FIRE → NVT → NPT) sharing one safety-hook chain, one logging protocol, and one checkpoint convention — the same primitives extend straight to alternative integrators (e.g. NVE, replica exchange) by swapping the integrator class.
- **Run a temperature sweep as a single multi-graph `Batch`** with per-graph `temperature` / `pressure` tensors broadcast via `batch_idx`, then de-interleave the resulting Zarr trajectory into per-graph trajectories with `extract_per_graph_trajectory`.
- **Extend an NPT run without thermostat / barostat transients** by persisting and pre-loading the integrator's internal state (`save_integrator_state` / `load_integrator_state` + manual `_state` assignment).
- **Diagnose the phase of a molecular-crystal trajectory** with the Yoneya–Harada $D + S_0$ classifier, separately for each half of an SLC system.

The same patterns apply to any single-component molecular crystal: change the CIF and the supercell, keep the rest of the pipeline.

## Extensions & Scaling

| Extension | Knob | What it adds |
|-----------|------|--------------|
| Multi-GPU sweep | `slc_multi_gpu.sh` + `--temps` flag | ~5× wall-time speedup for the SLC stage. |
| Larger cell | `SUPERCELL = (5, 5, 4)` ≈ 200 mol | Reduced finite-size bias on $T_\mathrm{m}$. |
| Long-range electrostatics | `USE_EWALD=True` | Accurate cell density for polar molecular crystals. |
| Different molecule | new CIF + verify `ATOMS_PER_MOL` | Same pipeline, new compound. |
| Bracketing $T_\mathrm{m}$ to ±5 K | finer `TEMPS` near the bracket | Quantitative $T_\mathrm{m}$ estimate. |
| Free-energy corroboration | thermodynamic integration (separate workflow) | Independent check on the SLC $T_\mathrm{m}$. |

## Summary

- **The full pipeline** (CIF → warmup → melt → SLC → $T_\mathrm{m}$) runs in one notebook with one foundation model — no compound-specific parameter fitting.
- **The Yoneya–Harada phase classifier** ($D$ + $S_0$) is the diagnostic backbone: every diagnostic cell in the notebook reads out one or both, separately for the crystal and melt halves.
- **Anisotropic NPT, vacuum gap, and per-half molecule unwrapping** are the three implementation details that make the difference between a working SLC and a failed FIRE2 within the first 1000 steps.
- **FAST_DEMO** lets students walk the analysis without committing to a 6 GPU-hour run; flipping it to `False` reproduces the cached trajectories.

## Discussion Prompts

1. **The five-temperature sweep returns a wide bracket on $T_\mathrm{m}$.** What sources of variance dominate at this system size — finite-size effects, simulation-time, model underbinding, or interface-roughness statistics? How would you design a follow-up sweep to disentangle them?
2. **The crystal-half $S_0$ at high $T$ does not always go to 0** (some axes retain residual order). What does this say about how naphthalene melts — does the crystal melt isotropically or does one principal axis stay ordered longer?
3. **Could you skip the `melt_naphthalene.py` stage and build the liquid half from scratch** (e.g., random packing, then NVT)? What invariant must the synthetic liquid share with the warm crystal for the SLC stack to work?
4. **The `_state` preload trick** preserves NHC + barostat momenta across notebook reruns. What goes wrong if you skip it (start every NPT extension with zero-init thermostat chains)? When wouldn't this matter?
5. **AIMNet2-2025 vs. AIMNet2 vs. AIMNet2 + Ewald.** For which compound classes would each be the correct default? What signal in the warmup diagnostics would make you switch?

## References

[1] L. Schmidt, D. Van der Spoel, M.-M. Walz, *ACS Phys. Chem. Au* **3**, 84–93 (2023). DOI: [10.1021/acsphyschemau.2c00045](https://doi.org/10.1021/acsphyschemau.2c00045) — Probing phase transitions in organic crystals with atomistic MD; SLC + rotational order parameter $S_0$ + diffusion coefficient $D$ as the screening procedure followed throughout this notebook.

[2] C. P. Brock & J. D. Dunitz, *Acta Crystallogr. B* **38**, 2218–2228 (1982). DOI: [10.1107/S0567740882008358](https://doi.org/10.1107/S0567740882008358) — Temperature dependence of thermal motion in crystalline naphthalene; the reference structure (CSD refcode NAPHTA10, CCDC 1216816) we use as the input CIF.

[3] D. M. Anstine, R. Zubatyuk, O. Isayev, *Chem. Sci.* (2025). DOI: [10.1039/D4SC08572H](https://doi.org/10.1039/D4SC08572H) — AIMNet2: a charge-equivariant neural-network potential for organic and elemental-organic molecules. The `aimnet2` and `aimnet2_2025` checkpoints loaded by `AIMNet2Wrapper` come from this work.

[4] M. P. Allen & D. J. Tildesley, *Computer Simulation of Liquids*, 2nd ed., Oxford University Press (2017). ISBN: 978-0-19-880319-5 — Standard reference for the Einstein-relation MSD-to-D fit, PBC-aware fractional unwrap of trajectory coordinates, and the COM-MSD diagnostic used in §5 / §10.